# Étiquetage automatique de questions posées sur StackOverflow

Projet n$^\text{o}$ 4 du [cursus Machine Learning Engineer][2] d'OpenClassrooms

Auteur : [Kiril ISAKOV][1]

Mentor : Nicolas TISSERAND

Projet démarré le 02/06/2025

[1]: https://github.com/kirisakow/
[2]: https://openclassrooms.com/fr/paths/794-machine-learning-engineer

## Énoncé

1. Requête SQL pour récupérer les données depuis [l'API StackExchange][1] :

    ```sql
    SELECT TOP 50000
      Title
      , Body
      , Tags
      , Score
      , AnswerCount
    FROM
      Posts
    WHERE
      PostTypeId = 1
      AND LEN(Tags) >= 1
      AND YEAR(CreationDate) = 2024
      AND MONTH(CreationDate) = 1 --1, 2, ..., 12
    ```

[1]: https://data.stackexchange.com/stackoverflow/query

## Étapes préliminaires

### Installer les librairies manquantes

In [ ]:
# !pip install -q --no-cache-dir numpy==1.26.4
# !pip uninstall -q -y pandas gensim pyLDAvis
# !pip install -q --no-cache-dir pandas gensim pyLDAvis
# !pip install --upgrade --no-cache-dir jupyter-client
!pip install --quiet --upgrade mlflow wandb transformers torch==2.9.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.5/788.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207

### Imports et constantes

In [ ]:
from bs4 import BeautifulSoup
from collections import Counter
from collections.abc import Callable
from functools import wraps
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
# from google.colab import drive
from io import StringIO
from pathlib import Path, PosixPath
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.exceptions import ConvergenceWarning, InconsistentVersionWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
# FutureWarning: Class PassiveAggressiveClassifier is deprecated in version 1.8 and will be removed in 1.10. Use
# `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier, SGDClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.utils._testing import ignore_warnings
from tqdm.notebook import tqdm, trange
from typing import List, Union, Any
import ast
import datetime as dt
import gc
import hashlib
import itertools as it
import joblib
import logging
import math
import mlflow
import mlflow.sklearn
import multiprocessing as mp
import nltk
import numpy as np
import os
import pandas as pd
import pickle
import pytz
import re
import seaborn as sns
import sys
import time
import torch
import transformers
import wandb
import warnings

# nltk.download('omw-1.4')
# nltk.download('punkt_tab')
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')

sns.set()

PROJECT_DIR = '2025.06.02.OC.proj.04.NLP.StackOverflow.tags'
DB_FILENAME_GLOB = 'so.2024.*.csv'
VECTORIZER_TOKEN_REGEX_PATTERN = r'\b[a-zA-Z0-9.][a-zA-Z0-9_+#~.-]*[a-zA-Z0-9+#]'
TAGS_SEPARATOR_REGEX_PATTERN = re.compile(r'[<>]')
WANDB_API_KEY = '4c856baf4dc7314db3f885f12d52e9bf6873d55f'
LOGGER_FORMAT = '%(asctime)s [%(levelname)s] %(message)s'

logging.Formatter.converter = lambda *_: dt.datetime.now(pytz.timezone('Europe/Paris')).timetuple()
logging.basicConfig(level=logging.INFO, format=LOGGER_FORMAT, force=True)
logr = logging.getLogger(__name__)
logr.setLevel(logging.INFO)
log_file_handler = logging.FileHandler(f"{dt.date.isoformat(dt.datetime.now(pytz.timezone('Europe/Paris')))}.log")
log_file_handler.setLevel(logging.DEBUG)
log_cons_handler = logging.StreamHandler()
log_cons_handler.setLevel(logging.ERROR)
log_file_handler.setFormatter(LOGGER_FORMAT)
log_cons_handler.setFormatter(LOGGER_FORMAT)
logr.addHandler(log_file_handler)
logr.addHandler(log_cons_handler)

logging.getLogger('alembic').setLevel(logging.WARNING)

### Fonctions

In [ ]:
def connect_drive(drive_path: str ='/content/mnt/MyDrive',
                  nb_link_path: str ='/content/notebooks'):
    if not os.path.ismount('/content/mnt'):
        drive.mount('/content/mnt')
    if not os.path.lexists(nb_link_path):
        os.symlink(src=drive_path + '/Colab Notebooks', dst=nb_link_path)
        sys.path.insert(0, nb_link_path)

def read_nth_line(path: Union[str, Path], n: Union[int, List[int]]=1) -> str:
    """Read `n`th line or a `[n, n']` range of lines from the `path` to a text file.
    First line is both n=1 and n=0."""
    frm, to = n if isinstance(n, list) else (max(n, 1), max(n, 1))
    ret = ''
    with open(path, 'rt') as txt_rdr:
        for i, line in enumerate(txt_rdr, 1):
            if i in range(frm, to + 1):
                ret += line
    return ret

def read_header(path: Union[str, Path]) -> str:
    return read_nth_line(path)

def strip_tags_of_a_chunk(chunk: pd.DataFrame):
    logr = globals().get('logr')
    logr.info('Processing one of the chunks...')
    for i in tqdm(chunk.index, desc="Processing the rows of one of the chunks", unit='row', leave=False):
        chunk.at[i, 'tags'] = tuple(tag for tag in TAGS_SEPARATOR_REGEX_PATTERN.split(chunk.at[i, 'tags']) if tag != '')
    return chunk

def concat_body_and_title_of_a_chunk(chunk: pd.DataFrame):
    logr = globals().get('logr')
    logr.info('Processing one of the chunks...')
    for i in tqdm(chunk.index, desc="Processing the rows of one of the chunks", unit='row', leave=False):
        chunk.at[i, 'Body'] = BeautifulSoup(chunk.at[i, 'Body'], 'html.parser').get_text(strip=True).replace('\n', ' ')
        chunk.at[i, 'Body'] = chunk.at[i, 'Title'] + ' ' + chunk.at[i, 'Body']
    return chunk

def imap_unordered_worker(chunks: List[pd.DataFrame], func, path: Path, logr: logging.Logger):
    num_cores = mp.cpu_count()
    with mp.Pool(num_cores) as pool:
        logr.info(f"Processing {path.name!r} by chunks...")
        for result in pool.imap_unordered(func, chunks):
            logr.info(f"Yield one of the processed chunks")
            yield result

def strip_tags_with_multiprocessing(path: Path, logr: logging.Logger, save_result: bool = True):
    """À l'aide de la parallélisation et grâce au module `multiprocessing`,
    - Enlever les chevrons '<' et '>' qui entourent chaque tag. """
    df = pd.read_csv(path.name)
    logr.info(f"Loaded {len(df)} rows from {path.name!r}")
    num_cores = mp.cpu_count()
    warnings.filterwarnings('ignore')
    chunks = np.array_split(df, num_cores)
    logr.info(f"{path.name!r} contents has been split into {len(chunks)} chunks to be processed by {num_cores} available CPU cores")
    chunks_processed = imap_unordered_worker(
            chunks, strip_tags_of_a_chunk, path, logr)
    df = pd.concat([*chunks_processed])
    warnings.filterwarnings('default')
    logr.info(f"Processed chunks have been concatenated")
    df = df.sort_index()
    logr.info(f"Dataset has been sorted by index")
    if save_result:
        df.to_csv(path.name, index=False)
        logr.info(f"{len(df)} rows saved as {path.name!r}")

def concat_body_and_title_with_multiprocessing(path: Path, logr: logging.Logger, save_result: bool = True):
    """À l'aide de la parallélisation et grâce au module `multiprocessing`,
    - Faire fusionner la colonne 'Title' avec 'Body'.
    - Supprimer la colonne 'Title'. """
    df = pd.read_csv(path.name)
    logr.info(f"Loaded {len(df)} rows from {path.name!r}")
    num_cores = mp.cpu_count()
    warnings.filterwarnings('ignore')
    chunks = np.array_split(df, num_cores)
    logr.info(f"{path.name!r} contents has been split into {len(chunks)} chunks to be processed by {num_cores} available CPU cores")
    chunks_processed = imap_unordered_worker(
            chunks, concat_body_and_title_of_a_chunk, path, logr)
    df = pd.concat([*chunks_processed])
    warnings.filterwarnings('default')
    logr.info(f"Processed chunks have been concatenated")
    df = df.sort_index()
    logr.info(f"Dataset has been sorted by index")
    cols_to_drop = ['Title']
    df = df.drop(columns=cols_to_drop)
    logr.info(f"Cols {cols_to_drop!r} have been dropped")
    df.columns = [c.lower() for c in df.columns]
    logr.info(f"Cols have been renamed to lowercase")
    if save_result:
        df.to_csv(path.name, index=False)
        logr.info(f"{len(df)} rows saved as {path.name!r}")

def compute_lda(csr_matr, *, min_topics, max_topics, step, perplexity_threshold):
    """- Train and evaluate LDA model's perplexity (the lower the better) for
    various topic sets of sizes in `range(min_topics, max_topics + 1, step)` over a
    sparse matrix which is a product of a `CountVectorizer.fit_transform(text_documents)`
    computation.
       - If perplexity < perplexity_threshold, then save (serialize) LDA model
    object as a pickle file."""
    for n_topics in trange(min_topics, max_topics + 1, step):
        lda = LatentDirichletAllocation(n_topics, random_state=42, n_jobs=-1)
        logr.info(f"Training LDA with {n_topics} topics...")
        lda.fit(csr_matr)
        perplexity = lda.perplexity(csr_matr)
        logr.info(f"Model with {n_topics} topics -> Perplexity: {perplexity:_.2f} (the lower the better)")
        if perplexity < perplexity_threshold:
            perplexity_threshold = perplexity
            fname = f'lda_{len(feature_names)}_words_{n_topics}_topics_perplexity_{perplexity:.2f}.pkl'
            joblib.dump(lda, fname)

def get_topic_top_words(lda_model, feature_names, n_top_words=10):
    """Return list of top words per topic as a dict {topic_id: [words...]}."""
    topic_keywords = {}
    for topic_idx, topic in enumerate(lda_model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_features_ind]
        topic_keywords[topic_idx] = top_words
    return topic_keywords

@ignore_warnings(category=InconsistentVersionWarning)
def topics_tag_documents(lda_model, csr_matr, feature_names, topic_threshold=0.2, top_words=5):
    """Assign 'predicted_tags' to each document based on topic distribution.
    ⚠️ Max tags = 1 / topic_threshold × top_words ;
        e. g. 1 / 0.2 × 5 = 25"""
    # Topic-word dictionary
    topic_keywords = get_topic_top_words(lda_model, feature_names, n_top_words=top_words)

    # Document-topic distribution
    doc_topic_dist = lda_model.transform(csr_matr)

    tags_per_doc = []
    for _, topic_probs in enumerate(doc_topic_dist):
        tags = []
        for topic_idx, prob in enumerate(topic_probs):
            if prob >= topic_threshold:  # topic is relevant
                tags.extend(topic_keywords[topic_idx])
        # remove duplicates, keep order
        tags_per_doc.append(list(dict.fromkeys(tags)))
    return tags_per_doc

def extract_tags(col: pd.Series,
                 separator_pattern: re.Pattern = TAGS_SEPARATOR_REGEX_PATTERN
                 ):
    for tags in col:
        yield tuple(tag for tag in separator_pattern.split(tags) if tag != '')

def get_variable_sizes():
    variables_info = []
    for name, obj in globals().items():
        if not name.startswith('_'):
            size_bytes = sys.getsizeof(obj)

            # For DataFrames, get more accurate memory usage
            if hasattr(obj, 'memory_usage'):
                size_bytes = obj.memory_usage(deep=True).sum()

            variables_info.append({
                'Variable': name,
                'Type': type(obj).__name__,
                'Size (bytes)': size_bytes,
                'Size (MB)': size_bytes / (1024 * 1024)
            })

    df = pd.DataFrame(variables_info)
    return df.sort_values('Size (bytes)', ascending=False)

def filtre_by_top_k_tags(df_sample: pd.DataFrame,
                         top_k_tags: tuple[str],
                         sample_size: int) -> pd.DataFrame:
    logr.info("Copy the original DF so it is not altered.")
    df_sample = df.copy()
    logr.info("Filter each question's tags by keeping only those in the top_k_tags list. "
              "Drop questions with no tags left.")
    df_sample.loc[:, 'tags'] = df_sample['tags'].apply(lambda tags_tuple: tuple(tag for tag in tags_tuple if tag in top_k_tags))
    df_sample = df_sample.loc[df_sample['tags'].apply(len) > 0]
    if len(df_sample) > sample_size:
        logr.info(f"{len(df_sample) = :,} > {sample_size = :,} \N{Rightwards Double Arrow} Reduce sample size down to {sample_size = :,}")
        df_sample = df_sample.sample(sample_size, random_state=42)
    return df_sample.reset_index(drop=True)

@ignore_warnings(category=ConvergenceWarning)
def trn_and_log_mdl(X_tr_preprocessed: Union[pd.DataFrame, pd.Series] = None,
                    y_tr: pd.Series = None,
                    model_recipe: Any = None,
                    grid_params: dict = None,
                    experiment_name: str = None,
                    project_name: str = None,
                    ):
    for parm, parm_name in zip((grid_params, y_tr), ('grid_params', 'y_tr')):
        if parm is None:
            raise ValueError(f"{parm_name} must be provided")

    mlflow.set_experiment(experiment_name)
    run = wandb.init(name=experiment_name, project=project_name) if project_name else wandb.init(name=experiment_name)

    with mlflow.start_run():
        grid = GridSearchCV(
            model_recipe,
            grid_params,
            return_train_score=True,
            scoring='f1_macro',
            cv=5,
            n_jobs=-1,
            verbose=3,
        )
        gc.collect()
        grid.fit(X_tr_preprocessed, y_tr)
        display(pd.DataFrame(grid.cv_results_))
        best_index = np.where(grid.cv_results_['rank_test_score'] == 1)[0][0]
        mlflow.log_metric('mean_train_score', grid.cv_results_['mean_train_score'][best_index])
        run.log({'mean_train_score': grid.cv_results_['mean_train_score'][best_index]})
        mlflow.log_metric('mean_test_score', grid.cv_results_['mean_test_score'][best_index])
        run.log({'mean_test_score': grid.cv_results_['mean_test_score'][best_index]})
        mlflow.log_metric('mean_score_time', grid.cv_results_['mean_score_time'][best_index])
        run.log({'mean_score_time': grid.cv_results_['mean_score_time'][best_index]})
        mlflow.log_metric('mean_fit_time', grid.cv_results_['mean_fit_time'][best_index])
        run.log({'mean_fit_time': grid.cv_results_['mean_fit_time'][best_index]})
        mlflow.log_params(grid.best_params_)
        run.config.update(grid.best_params_)
        mlflow.sklearn.log_model(
            grid.best_estimator_,
            name=f'best_estimator_{experiment_name}',
            input_example=X_tr_preprocessed[:5]
        )
        joblib.dump(
            grid.best_estimator_, f'best_estimator_{experiment_name}.joblib'
        )

    run.finish()


### Connecter ce notebook à mon Google Drive

In [ ]:
connect_drive(drive_path='/content/mnt/MyDrive',
              nb_link_path='/content/notebooks')

Mounted at /content/mnt


### Se positionner dans le répertoire du projet

In [ ]:
%cd /content/notebooks/{PROJECT_DIR}
! ls -lhAap --group-directories-first

/content/mnt/MyDrive/Colab Notebooks/2025.06.02.OC.proj.04.NLP.StackOverflow.tags
total 1.7G
drwx------ 2 root root 4.0K Jan  5 08:41 mlruns/
drwx------ 2 root root 4.0K Oct 27 08:21 wandb/
-rw------- 1 root root    0 Aug 25 10:20 2025-08-25.log
-rw------- 1 root root  960 Sep  8 14:16 2025-09-08.log
-rw------- 1 root root    0 Sep 15 07:22 2025-09-15.log
-rw------- 1 root root 163K Sep 22 21:22 2025-09-22.log
-rw------- 1 root root    0 Oct 13 18:07 2025-10-13.log
-rw------- 1 root root    0 Oct 20 07:42 2025-10-20.log
-rw------- 1 root root    0 Oct 27 08:31 2025-10-27.log
-rw------- 1 root root 1.5K Nov 10 23:24 2025-11-10.log
-rw------- 1 root root    0 Dec  1 21:15 2025-12-01.log
-rw------- 1 root root  680 Jan  8 14:16 2026-01-08.log
-rw------- 1 root root  160 Jan 12 08:33 2026-01-12.log
-rw------- 1 root root  160 Jan 19 08:40 2026-01-19.log
-rw------- 1 root root  920 Jan 26 08:29 2026-01-26.log
-rw------- 1 root root 345K Jan  8 10:04 best_estimator_Bert_100000_PassiveAggress

### Transformer les données (à faire une seule fois)

In [ ]:
paths = sorted(Path('data/').glob(DB_FILENAME_GLOB))
unit = 'file'
for path in tqdm(paths, desc="Processing files", unit=unit, leave=False):
    # concat body and title columns
    if (missing_col := 'Title').casefold() in read_header(path).casefold():
        logr.info(f"Going to concatenate 'Body' and 'Title' columns in file {path.name!r}")
        concat_body_and_title_with_multiprocessing(path, logr, save_result=True)
    else:
        logr.info(f"Column {missing_col!r} not found in file {path.name!r}... Skipping to the next {unit}")

    # strip the tags column of '<' & '>'
    df_mini = pd.read_csv(StringIO(read_nth_line(path, [1, 3])))
    if any(test_char in df_mini.loc[0, 'tags'] for test_char in ['<', '>']):
        logr.info(f"Going to strip the tags of the '<' and '>' delimiters in file {path.name!r}")
        strip_tags_with_multiprocessing(path, logr, save_result=True)
    else:
        logr.info(f"No '<' or '>' found in the 'tags' column in file {path.name!r}... Skipping to the next {unit}")

### Charger les données

In [4]:
df = None
for path in tqdm(sorted(Path('data/').glob(DB_FILENAME_GLOB)), leave=True, unit='file'):
    df_from_path = pd.read_csv(path.absolute(), converters={'tags': ast.literal_eval})
    df = pd.concat([df, df_from_path])
df = df.reset_index(drop=True)
df

  0%|          | 0/12 [00:00<?, ?file/s]

,body,tags,score,answercount
0,Make URL Address Text a Clickable URL Using se...,"(google-apps-script, pdf-generation)",2,1
1,7zip commandline extract excluding single file...,"(extract, 7zip)",0,1
2,How do I delete an element from an array and s...,"(javascript, arrays, google-chrome-extension, ...",0,0
3,Issues with JumpList in C# I am working on a C...,"(c#, jump-list)",0,0
4,Determine whether app launches AOT binary (or ...,"(android,)",1,0
...,...,...,...,...
420286,background-image in diagonal pattern When usin...,"(css,)",-1,1
420287,React lazy components with poor internet conne...,"(reactjs, code-splitting, react-lifecycle, rea...",1,1
420288,Row-Level Locking for Concurrent Processing in...,"(java, spring, spring-boot, multithreading, jpa)",0,0
420289,Prevent VSCode from deleting highlighted regio...,"(visual-studio-code, emacs)",0,0


## Étape 1 : exploration rapide

nombre de tags uniques :

In [ ]:
distinct_tags = set(it.chain.from_iterable(df['tags']))
print(f"{len(distinct_tags) = }")

len(distinct_tags) = 35371


distribution des tags :

In [ ]:
bag_of_tags = Counter(it.chain.from_iterable(df['tags']))
bag_of_tags = dict(sorted(bag_of_tags.items(), reverse=True, key=lambda item: item[1]))
top_n = 100
print(f'Top {top_n} tags:\n')
dict(list(bag_of_tags.items())[:top_n])

Top 100 tags:



{'python': 56005,
 'javascript': 32089,
 'c#': 22260,
 'java': 19789,
 'reactjs': 18801,
 'android': 16440,
 'c++': 13009,
 'html': 12998,
 'flutter': 12756,
 'r': 12685,
 'typescript': 12129,
 'css': 10646,
 'node.js': 10617,
 'php': 9852,
 'angular': 8776,
 'sql': 8546,
 'azure': 8145,
 'spring-boot': 7867,
 'next.js': 7514,
 'excel': 7311,
 'docker': 6856,
 '.net': 6732,
 'c': 6650,
 'ios': 6362,
 'kotlin': 6322,
 'swift': 6171,
 'postgresql': 6145,
 'react-native': 6038,
 'pandas': 6013,
 'amazon-web-services': 5984,
 'python-3.x': 5693,
 'django': 5340,
 'laravel': 5208,
 'dart': 4672,
 'vba': 4652,
 'asp.net-core': 4498,
 'json': 4492,
 'visual-studio-code': 4392,
 'spring': 4385,
 'swiftui': 4344,
 'sql-server': 4052,
 'powershell': 3973,
 'firebase': 3918,
 'dataframe': 3837,
 'windows': 3757,
 'mysql': 3730,
 'rust': 3660,
 'arrays': 3345,
 'linux': 3335,
 'vue.js': 3278,
 'wordpress': 3181,
 'git': 3123,
 'android-jetpack-compose': 2938,
 'selenium-webdriver': 2890,
 'mongodb

## Étape 2 : Appliquer des méthodes d’extraction de features spécifiques des données textuelles

### NLTK : Lemmatization & Stemming (🇫🇷 lemmatisation et racinisation)

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Sample text
random_idx = np.random.randint(0, df.shape[0])
text = df['body'][random_idx]

# 1. Tokenization
tokens = word_tokenize(text)

# 2. Lowercase
tokens = [t.lower() for t in tokens]

# 3. Remove Stopwords
stop_words = set(stopwords.words('english'))
filtered_tokens = [word for word in tokens if word.isalpha() and word not in stop_words]

# 4. Lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(token) for token in filtered_tokens]

# 5. Stemming
stemmer = PorterStemmer()
stemmed = [stemmer.stem(token) for token in filtered_tokens]

# Output
print("Original Tokens:", tokens, end='\n\n')
print("Filtered Tokens (no stopwords):", filtered_tokens, end='\n\n')
print("Lemmatized:", lemmatized, end='\n\n')
print("Stemmed:", stemmed, end='\n\n')


Original Tokens: ['does', 'azure', 'cosmos', 'db', 'support', 'mongodb', 'with', 'in', 'memory', 'cache', 'we', 'have', 'been', 'using', 'mongodb', 'on-prem', 'as', 'an', 'in-memory', 'cache', 'database', 'with', 'a', 'replica', 'setup', '.', 'we', 'are', 'planning', 'to', 'move', 'to', 'azure', 'and', 'use', 'cosmos', 'db', 'which', 'supports', 'mongodb', 'api', 'and', 'drivers', 'so', 'that', 'we', 'do', 'not', 'require', 'any', 'code', 'changes', '.', 'does', 'cosmos', 'db', 'support', 'mongodb', 'with', 'an', 'in', 'memory', 'cache', '?']

Filtered Tokens (no stopwords): ['azure', 'cosmos', 'db', 'support', 'mongodb', 'memory', 'cache', 'using', 'mongodb', 'cache', 'database', 'replica', 'setup', 'planning', 'move', 'azure', 'use', 'cosmos', 'db', 'supports', 'mongodb', 'api', 'drivers', 'require', 'code', 'changes', 'cosmos', 'db', 'support', 'mongodb', 'memory', 'cache']

Lemmatized: ['azure', 'cosmos', 'db', 'support', 'mongodb', 'memory', 'cache', 'using', 'mongodb', 'cache', '

Conclusions :

- Le stemming (la racinisation) n'est clairement pas pertinent.

- La lemmatisation n'est finalement pas très pertinente non plus étant donné la particularité du vocabulaire, de la syntaxe et de la ponctuation spécifiques aux langages de programmation où nombre de mots et d'usages n'existent pas dans l'anglais standard (du `snake_case`, `UPPER_SNAKE_CASE`, `camelCase`, `PascalCase`, `kebab-case`, etc).

- Les stopwords devraient très clairement être personnalisés afin d'éviter de laisser passer à la trappe des stopwords autrement banales mais qui sont pourtant chargés de sens étant donné la spécificité du champ lexical de la programmation (de faux stopwords comme `for`, `if`, `else`, `do`, `while`, `in`, `any`, `not`, etc).

### Échantillonnage du DataFrame

L'idée est d'obtenir une matrice creuse prête à être utilisée.

La conversion en DataFrame dense est possible mais attention à la surconsommation de la mémoire vive, la RAM !

In [12]:
df_sample = df.sample(n=100_000, random_state=42)

Pour commencer, entraîner un vectorizer sans filtres :

In [ ]:
vectorizer_no_filter = CountVectorizer()
X_no_filter = vectorizer_no_filter.fit_transform(df_sample['body'])
print(f"Vocabulaire sans filtres pour un échantillon de {df_sample.shape[0]:_} documents : {len(vectorizer_no_filter.vocabulary_):_} mots")

Vocabulaire sans filtres pour un échantillon de 100_000 documents : 1_004_918 mots


Entraîner un vectorizer filtré :

In [9]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd
import numpy as np
import pickle

csr_matr_filename, vectorizer_filtered_filename = 'filtered_vocab_csr_matr.pkl', 'vectorizer_filtered.pkl'
if not os.path.exists(f'models/{csr_matr_filename}') and not os.path.exists(f'models/{vectorizer_filtered_filename}'):
    vectorizer_filtered = CountVectorizer(min_df=50, max_df=0.2000, lowercase=True,
                                          token_pattern=VECTORIZER_TOKEN_REGEX_PATTERN)
    filtered_vocab_csr_matr = vectorizer_filtered.fit_transform(df_sample['body'])
    for fname, obj in zip([csr_matr_filename, vectorizer_filtered_filename],
                          [filtered_vocab_csr_matr, vectorizer_filtered]):
        joblib.dump(obj, fname)
else:
    for fname, varname in zip([csr_matr_filename, vectorizer_filtered_filename],
                              [csr_matr_filename.split('.')[0], vectorizer_filtered_filename.split('.')[0]]):
        globals()[varname] = joblib.load(f'models/{fname}')

# Informations sur la vectorisation
print(f"Forme de la matrice : {filtered_vocab_csr_matr.shape}")
print(f"Type de la matrice : {type(filtered_vocab_csr_matr).__qualname__}")
print(f"Densité de la matrice : {filtered_vocab_csr_matr.nnz / (filtered_vocab_csr_matr.shape[0] * filtered_vocab_csr_matr.shape[1]):.4f}")
print()
print(f"Vocabulaire total (nombre de features) après filtrage : {len(vectorizer_filtered.vocabulary_):,}")
print()
feature_names = vectorizer_filtered.get_feature_names_out()
n = 100
print(f"{n} features au hasard : {np.random.choice(feature_names, n)}")

Forme de la matrice : (100000, 11384)
Type de la matrice : csr_matrix
Densité de la matrice : 0.0056

Vocabulaire total (nombre de features) après filtrage : 11,384

100 features au hasard : ['contour' 'thickness' '2b' 'v2.0' 'silver' 'noted' 'dbset' 'easier'
 'entitymanager' 'emotion' 'valueerror' 'eks' 'dolore' 'rectangles'
 'it.is' 'p-1' 'conflict' 'series' 'space-between' 'linestyle' 'quit'
 'math.floor' 'outputtype' 'mocks' 'tracker' 'verb' 'automatic'
 'text-transform' 'enumeration' 'converts' '161' 'composables' 'cycle'
 'background-image' 'involved' 'codebase' 'thus' 'mem' 'chars' 'disappear'
 'linker' '281' 'tooltip' 'navigate' 'crashing' 'stdout' 'fortawesome'
 'jsfiddle.net' 'mime' 'error.message' 'rebase' 'hot' 'foreignkey' 'pic'
 'mainactivity' 'manual' 'suffix' 'sink' 'concept' 'technique' 'ser'
 'ws.range' 'linkedin' 'mutable' '4.4' 'databasename' 'express.json' 'c3'
 'designated' 'keys' 'sympy' 'timedelta' 'gke' 'contexts' 'border-style'
 'claim' 'd+' 'covering' 'http_h

/home/mira/.cache/pypoetry/virtualenvs/oc-stackoverflow-api-c5RXNrAM-py3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


### Approche non-supervisée : LDA (Latent Dirichlet allocation)

- https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html

- https://www.baeldung.com/cs/topic-modeling-coherence-score

- https://www.linkedin.com/advice/1/how-do-you-evaluate-coherence-perplexity

Recherche du nombre optimal de topics :
- algorithme utilisé : `sklearn.decomposition.LatentDirichletAllocation` ;
- métrique de performance : perplexity.

In [ ]:
import math
import time
from sklearn.decomposition import LatentDirichletAllocation

fname = 'filtered_vocab_csr_matr.pkl'
filtered_vocab_csr_matr = joblib.load(f'models/{fname}')

try:
    compute_lda(filtered_vocab_csr_matr, min_topics=130, max_topics=200, step=10, perplexity_threshold=1216)
except KeyboardInterrupt:
    logr.warning("Traitement interrompu manuellement")

In [6]:
BEST_LDA_FILENAME = 'lda_11384_words_120_topics_perplexity_1216.72.pkl'

Description des topics

In [10]:
def print_top_words(model, feature_names, n_top_words=10):
    """Print the top words for each topic in a fitted sklearn LDA model."""
    print(f"\n=== Top {n_top_words} words per topic for an LDA trained for {model.n_components} topics ===\n")
    for topic_idx, topic in enumerate(model.components_, start=1):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_features_ind]
        top_words = sorted(top_words)
        print(f"Topic {topic_idx: >2}: {', '.join(top_words)}", end='\n\n')

# Assuming you already have vectorizer and lda model trained:
# feature_names = vectorizer_filtered.get_feature_names_out()
fname = BEST_LDA_FILENAME
best_lda = joblib.load(f'models/{fname}')
print_top_words(best_lda, feature_names, n_top_words=10)


=== Top 10 words per topic for an LDA trained for 120 topics ===

Topic  1: cpu, every, minutes, performance, seconds, speed, start, time, timer, times

Topic  2: admin, args, django, expression, kwargs, lang, params, path, render, request

Topic  3: 10, column, font, items, list, pos, root, row, text, tkinter

Topic  4: binding, control, grid, label, maui, property, setter, text, xaml, xmlns

Topic  5: 0.0, bundle, com, could, find, found, libs, session, sessions, snapshot

Topic  6: camera, fill, http, num, path, score, stream, streaming, streams, svg

Topic  7: also, just, me, one, other, ve, was, way, work, you

Topic  8: appdata, last, lib, line, local, most, python, recent, site-packages, users

Topic  9: bean, boot, configuration, import, password, public, security, spring, step, username

Topic 10: app, component, components, const, export, import, native, react, style, view

Topic 11: add, begin, cart, end, form, order, payment, quantity, stripe, then

Topic 12: app, controll

Attribution des mots-clés

In [16]:
import numpy as np

try:
    fname = BEST_LDA_FILENAME
    best_lda = joblib.load(f'models/{fname}')
    df_sample['predicted_tags'] = topics_tag_documents(best_lda, filtered_vocab_csr_matr, feature_names, topic_threshold=0.2, top_words=5)
except KeyboardInterrupt:
    logr.warning("Traitement interrompu manuellement")

/home/mira/.cache/pypoetry/virtualenvs/oc-stackoverflow-api-c5RXNrAM-py3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LatentDirichletAllocation from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [23]:
def truncated(string, n_chars, ending='...'):
    return string[:n_chars] + ending if len(string) >= n_chars + len(ending) else string


demo_sample = df_sample[['body', 'tags', 'predicted_tags']].sample(n=5)
_max_colwidth = pd.options.display.max_colwidth
pd.options.display.max_colwidth = None
display(
    pd.DataFrame({
        'Expected tags:': demo_sample['tags'],
        'Predicted tags:': demo_sample['predicted_tags'].apply(tuple),
        'Input (title + body, truncated):': demo_sample['body'].apply(lambda question_body: truncated(question_body, 300)),
    })
)
pd.options.display.max_colwidth = _max_colwidth

,Expected tags:,Predicted tags:,"Input (title + body, truncated):"
311113,"(c#, blazor, mudblazor)","(public, string, var, set, await)","Dynamic Controls in MudBlazor/Blazor I am trying to create dynamic controls in MudBlazor. I am able to do that using Dictionary, as you can see in the code below. The problem I am facing is while removing a control. When Test3 is removed, one control goes away from the display but it looks like the ..."
294118,"(serialization, azure-functions, azureservicebus, .net-8.0, azure-functions-isolated)","(input, print, loop, enter, output)",JObject Serialization issues in Azure Functions Isolated process with Service Bus connection I have Azure Function running on .NET 8 with Isolated process type. I have Service Bus queue trigger to listenqueueAand after processing I want to put message to other queuequeueB.I have code like this:[Func...
121398,"(python-polars,)","(install, build, installed, bin, tmp, each, filter, first, list, one)","polars groupby custom function on multiple columns I have a prediction vs actual dataset over different time periods/segments. I would like to evaluate the prediction using polars groupby, but couldn't figure out what's the best way to do it. In pandas, it would look likedf.groupby(""seg"").apply(lamb..."
23197,"(flutter, dart, gridview, custom-scrolling, customscrollview)","(time, seconds, speed, start, performance, way, just, was, me, one, connection, server, connect, client, port)",why the top widget in custom scroll view doesn't scroll I'm trying to use custom scroll view and the slivers are a container that have a pic and gridview when I try to scroll down or up just the list view is moving and the container does notthis is my code`import 'package:flutter/material.dart'; im...
401463,"(java, spring-boot, swagger)","(table, insert, sql, into, tables, column, row, columns, rows, values)",org.springframework.web.servlet.mvc.method.annotation.ExceptionHandlerExceptionResolver Added the whole POM file and also swagger dependencies.I started facing this issue once upgraded to 3.x from 2.x also I am facing the gson seriable issues org.springframework.beans.factory.BeanCreationException: ...


Conclusions :

- pas scalable : trop d'effort et de temps humain et de calcul pour un résultat insatisfaisant ;

- métrique possible pour évaluer quantitivement la pertinence des topics vs tags données (équivalente à une évaluation par un expert métier) :

    - à supposer que les tags de base sont (a) disponibles à la consultation, (b) fiables et (c) *idéalement* figurant parmi les mots présents dans chaque document (⚠️ ce qui n'est pas forcément le cas en réalité) :
  
        - évaluer Precision et Recall entre les topic et les tags pour chacun des doc ;

        - évaluer la similarité sémantique entre les topic et les tags pour chacun des docs

    - à supposer que les tags de base ne sont ni disponibles à la consultation ni fiables :

        - évaluer (noter) à la main (càd par un groupe d'experts) les topics prédits mais aussi les tags de base ; comparer les notes ;

        - tester la stabilité de l'approche même en entraînant, pour chaque document, plusieurs fois un LDA différent selon son paramètre `random_state`

### Approche supervisée

cf le notebook correspondant